In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.csp import CSP
from src.constraints import BinaryConstraint
from src.solvers import (
    backtracking_search,
    backtracking_search_mrv,
    backtracking_search_mrv_degree,
    backtracking_search_with_stats,
    backtracking_search_mrv_with_stats,
    backtracking_search_mrv_degree_with_stats,
    backtracking_search_forward_checking,
    backtracking_search_dynamic_mrv_forward_checking,
    backtracking_search_dynamic_fc_with_stats,
    backtracking_search_ac3,
    backtracking_search_ac3_with_stats,
    backtracking_search_lcv,
    backtracking_search_lcv_with_stats,
)
from src.generators import (
    generate_graph_coloring_csp
)
from src.experiments import (
    run_solver_experiment
)
from src.constraint_propagation import (
    ac3    
)

<h1> Step 9: Total Comparisons </h1>

<h3> MRV + Degree + LCV + Forward Checking </h3>

Variable Selection:</br>
Dynamic MRV
+
Degree


Value Selection:</br>
LCV


Propagation:</br>
Forward Checking

In [ ]:
random_problem = generate_graph_coloring_csp(
    num_variables=20,
    num_colors=4,
    edge_probability=0.2,
)

random_problem

In [ ]:
solution_lcv = backtracking_search_lcv(
    random_problem
)

In [ ]:
random_problem.is_consistent(
    solution_lcv
)

True

| Algorithm                       | Variable Selection   | Value Ordering | Propagation      |
| ------------------------------- | -------------------- | -------------- | ---------------- |
| Naive                           | First Variable       | First Value    | None             |
| MRV + Degree                    | MRV + Degree         | First Value    | None             |
| Dynamic MRV + FC                | Dynamic MRV + Degree | First Value    | Forward Checking |
| AC3 + Dynamic MRV + FC          | Dynamic MRV + Degree | First Value    | AC-3 + FC        |
| Dynamic MRV + Degree + LCV + FC | Dynamic MRV + Degree | LCV            | Forward Checking |


Now, for a fair comparison, all solvers should:

Run on the same CSP.

Tested with several different trials.

Why?

Because the generator is random.

In [ ]:
benchmark_solvers = [
    (
        "Naive",
        backtracking_search_with_stats
    ),
    (
        "MRV+Degree",
        backtracking_search_mrv_degree_with_stats
    ),
    (
        "Dynamic MRV+FC",
        backtracking_search_dynamic_fc_with_stats
    ),
    (
        "AC3+Dynamic MRV+FC",
        backtracking_search_ac3_with_stats
    ),
    (
        "Dynamic MRV+Degree+LCV+FC",
        backtracking_search_lcv_with_stats
    ),
]

In [ ]:
final_results = []
total_trials = 1

for trial in range(total_trials):
    problem = generate_graph_coloring_csp(
        num_variables=30,
        num_colors=4,
        edge_probability=0.2,
    )

    for name, solver in benchmark_solvers:
        result = run_solver_experiment(
            problem,
            solver,
        )

        result["Algorithm"] = name
        result["Trial"] = trial

        final_results.append(
            result
        )

In [ ]:
final_df = pd.DataFrame(
    final_results
)

final_df

,Nodes Visited,Assignments Tried,Backtracks,Success,Execution Time,Algorithm,Trial
0,1503435,6013683,1503404,True,78.094800,Naive,0
1,379,1466,348,True,0.021043,MRV+Degree,0
2,54,60,23,True,0.011953,Dynamic MRV+FC,0
3,54,60,23,True,0.022450,AC3+Dynamic MRV+FC,0
4,54,60,23,True,0.017387,Dynamic MRV+Degree+LCV+FC,0


### Benchmark Analysis

The results demonstrate the significant impact of heuristic-guided search in CSP solving. Naive backtracking explored more than 1.5 million nodes due to uninformed search, while MRV and Degree heuristics dramatically reduced the search space. Adding Forward Checking further improved efficiency by pruning inconsistent branches early, reducing the number of visited nodes from hundreds to tens. In this experiment, AC-3 did not provide additional pruning benefits because the graph-coloring constraints allowed most domain values to remain arc-consistent. Similarly, LCV did not significantly change the search behavior, indicating that its effectiveness depends on the structure of the constraint network.

In [ ]:
final_summary = (
    final_df
    .groupby("Algorithm")
    [
        [
            "Nodes Visited",
            "Assignments Tried",
            "Backtracks",
            "Execution Time",
            "Success",
        ]
    ]
    .mean()
    .reset_index()
)

final_summary

,Algorithm,Nodes Visited,Assignments Tried,Backtracks,Execution Time,Success
0,AC3+Dynamic MRV+FC,54.0,60.0,23.0,0.022450,1.0
1,Dynamic MRV+Degree+LCV+FC,54.0,60.0,23.0,0.017387,1.0
2,Dynamic MRV+FC,54.0,60.0,23.0,0.011953,1.0
3,MRV+Degree,379.0,1466.0,348.0,0.021043,1.0
4,Naive,1503435.0,6013683.0,1503404.0,78.094800,1.0


### Stres Test

In [ ]:
large_problem = generate_graph_coloring_csp(
    num_variables=90,
    num_colors=7,
    edge_probability=0.2,
)

In [ ]:
large_benchmark_solvers = [
    (
        "Dynamic MRV+FC",
        backtracking_search_dynamic_fc_with_stats
    ),
    (
        "AC3+Dynamic MRV+FC",
        backtracking_search_ac3_with_stats
    ),
    (
        "Dynamic MRV+Degree+LCV+FC",
        backtracking_search_lcv_with_stats
    ),
]

In [ ]:
large_results = []

for name, solver in large_benchmark_solvers:
    solution, stats = solver(
        large_problem
    )
    result = stats.summary()
    result["Algorithm"] = name
    large_results.append(
        result
    )

In [ ]:
large_df = pd.DataFrame(
    large_results
)

large_df

,Nodes Visited,Assignments Tried,Backtracks,Algorithm
0,189,207,98,Dynamic MRV+FC
1,189,207,98,AC3+Dynamic MRV+FC
2,91,90,0,Dynamic MRV+Degree+LCV+FC


In [ ]:
solution, stats = backtracking_search_dynamic_fc_with_stats(
    large_problem
)

solution

{'X8': 0,
 'X80': 1,
 'X24': 2,
 'X25': 0,
 'X62': 3,
 'X42': 2,
 'X65': 1,
 'X26': 2,
 'X79': 4,
 'X0': 1,
 'X49': 3,
 'X78': 4,
 'X2': 5,
 'X7': 1,
 'X89': 2,
 'X21': 1,
 'X71': 4,
 'X67': 2,
 'X9': 3,
 'X69': 0,
 'X39': 3,
 'X50': 5,
 'X16': 3,
 'X1': 4,
 'X64': 5,
 'X72': 1,
 'X13': 0,
 'X77': 5,
 'X81': 4,
 'X52': 6,
 'X45': 2,
 'X35': 0,
 'X54': 3,
 'X76': 4,
 'X20': 0,
 'X63': 4,
 'X30': 2,
 'X66': 6,
 'X57': 4,
 'X15': 0,
 'X75': 5,
 'X6': 6,
 'X14': 6,
 'X53': 5,
 'X58': 2,
 'X56': 5,
 'X73': 6,
 'X12': 6,
 'X36': 4,
 'X33': 6,
 'X19': 0,
 'X59': 5,
 'X60': 1,
 'X48': 3,
 'X3': 4,
 'X11': 5,
 'X46': 6,
 'X70': 2,
 'X74': 3,
 'X68': 5,
 'X27': 0,
 'X17': 3,
 'X86': 6,
 'X44': 1,
 'X83': 4,
 'X38': 3,
 'X31': 6,
 'X88': 2,
 'X22': 4,
 'X28': 2,
 'X82': 5,
 'X84': 5,
 'X51': 0,
 'X55': 3,
 'X29': 1,
 'X5': 6,
 'X43': 1,
 'X37': 4,
 'X47': 4,
 'X87': 3,
 'X61': 3,
 'X32': 1,
 'X40': 0,
 'X34': 1,
 'X18': 2,
 'X85': 6,
 'X10': 2,
 'X23': 5,
 'X41': 2,
 'X4': 1}

### Large-Scale CSP Benchmark Analysis

A larger graph-coloring problem was evaluated to analyze the impact of advanced heuristics. Dynamic MRV combined with Forward Checking reduced unnecessary exploration but still required several backtracking operations due to non-optimal value ordering. Adding AC-3 did not improve the results because the graph-coloring constraints remained mostly arc-consistent before search. In contrast, incorporating the Least Constraining Value (LCV) heuristic significantly improved performance by selecting values that preserved more flexibility for neighboring variables, resulting in fewer explored nodes and eliminating backtracking in this experiment.